In [ ]:
# Only this cell fixes the numpy/scipy/sklearn clash. Do NOT import sklearn here — restart first.
import subprocess
import sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", *args], stdout=subprocess.DEVNULL)

pip("install", "-q", "-U", "pip")
for pkg in ("numpy", "scipy", "scikit-learn"):
    pip("uninstall", "-y", pkg)
pip("install", "-q", "--no-cache-dir", "numpy==2.2.6")
pip("install", "-q", "--no-cache-dir", "scipy==1.15.2", "scikit-learn==1.6.1")
pip("install", "-q", "fastapi", "uvicorn", "pyngrok", "nest-asyncio", "python-multipart")
pip("install", "-q", "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git")
pip("install", "-q", "--force-reinstall", "--no-deps", "numpy==2.2.6")
pip("install", "-q", "--no-cache-dir", "scipy==1.15.2", "scikit-learn==1.6.1")

print("Install finished.")
print(">>> Session → Restart session, then run cells 2 onward (skip this cell). <<<")

In [ ]:
import os

# Kaggle: Add-ons → Secrets → HF_TOKEN, NGROK_TOKEN, NGROK_DOMAIN
HF_TOKEN = os.environ.get('HF_TOKEN', '')
NGROK_TOKEN = os.environ.get('NGROK_TOKEN', '')
NGROK_DOMAIN = os.environ.get('NGROK_DOMAIN', '')  # hostname only, e.g. xxx.ngrok-free.dev

if not all((HF_TOKEN, NGROK_TOKEN, NGROK_DOMAIN)):
    raise ValueError('Set HF_TOKEN, NGROK_TOKEN, and NGROK_DOMAIN as Kaggle secrets or env vars')

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '300'

from pyngrok import conf
conf.get_default().auth_token = NGROK_TOKEN
print('Credentials set')

Credentials set


In [2]:
# Preflight: catches numpy/scipy mismatch before loading the ~7 GB model
import numpy as np
from scipy.sparse import issparse
from sklearn.model_selection import GroupShuffleSplit

from tribev2 import TribeModel

CACHE_DIR = '/kaggle/working/tribe_cache'
print('Loading TRIBE v2 — first run downloads ~7 GB, subsequent runs use cache...')
tribe_model = TribeModel.from_pretrained('facebook/tribev2', cache_folder=CACHE_DIR)
print('TRIBE v2 ready')

/usr/local/lib/python3.12/dist-packages/neuralset/extractors/base.py:707: UserWarning: LabelEncoder: event_types has not been set, are you sure you want to apply this extractor to all events?
  warnings.warn(
2026-05-19 00:21:22 - WARNING - neuralset.extractors.base:798 - Missing events will be encoded using the default all-zero value (for example, 0 or a zero vector/tensor), which may be indistinguishable from a valid class if that class is also mapped to zeros. Set treat_missing_as_separate_class=True to avoid this.


Loading TRIBE v2 — first run downloads ~7 GB, subsequent runs use cache...


config.yaml: 0.00B [00:00, ?B/s]

best.ckpt:   0%|          | 0.00/709M [00:00<?, ?B/s]

2026-05-19 00:21:29 - WARNING - neuralset.extractors.base:798 - Missing events will be encoded using the default all-zero value (for example, 0 or a zero vector/tensor), which may be indistinguishable from a valid class if that class is also mapped to zeros. Set treat_missing_as_separate_class=True to avoid this.
INFO - Loading model from /root/.cache/huggingface/hub/models--facebook--tribev2/snapshots/f894e783020944dcd96e5568550afe2aa9743f9f/best.ckpt
/usr/local/lib/python3.12/dist-packages/x_transformers/x_transformers.py:439: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
/usr/local/lib/python3.12/dist-packages/x_transformers/x_transformers.py:461: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)


TRIBE v2 ready


In [3]:
import hashlib
import os
import tempfile
from typing import Optional

import numpy as np
from fastapi import FastAPI, File, Form, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse, JSONResponse
from pydantic import BaseModel

try:
    from nilearn import datasets, plotting
    _NILEARN_IMPORT_ERROR = None
except Exception as e:
    datasets = None
    plotting = None
    _NILEARN_IMPORT_ERROR = str(e)

TRIBE_API_VERSION = '2.4'
VIEWER_DIR = '/kaggle/working/tribe_viewers'
os.makedirs(VIEWER_DIR, exist_ok=True)

app = FastAPI(title='TRIBE v2 Activation API', version=TRIBE_API_VERSION)
app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

VIDEO_EXT = {'.mp4', '.mov', '.avi', '.mkv', '.webm'}
AUDIO_EXT = {'.wav', '.mp3', '.flac', '.m4a', '.ogg', '.aac'}

_text_cache = {}  # sha256(text) -> (preds, segments)
_viewer_cache = {}  # analysis_id -> HTML file path

# Approximate fsaverage5 vertex slices from the original notebook.
# These are useful product/demo signals, not atlas-grade neuroscience regions.
REGION_MASKS = {
    'visual_cortex': (1000, 2000),
    'language_network': (3000, 4500),
    'attention': (5000, 6000),
    'emotional_response': (6500, 7500),
    'memory_encoding': (8000, 9000),
}


def _segments_json(segments) -> list:
    if segments is None:
        return []
    try:
        import pandas as pd
        if isinstance(segments, pd.DataFrame):
            return segments.to_dict(orient='records')
    except ImportError:
        pass
    if isinstance(segments, list):
        out = []
        for s in segments:
            if isinstance(s, dict):
                out.append(s)
            elif hasattr(s, 'start') and hasattr(s, 'end'):
                out.append({'start': float(s.start), 'end': float(s.end)})
            else:
                out.append({'value': str(s)})
        return out
    return [{'value': str(segments)}]


def _viewer_preds(activation: np.ndarray) -> list:
    """Per-frame 0–1 normalization (|value|) for BrainViewer.jsx."""
    out = []
    for frame in np.asarray(activation, dtype=np.float64):
        v = np.abs(frame)
        mx = float(v.max())
        out.append((v / mx).tolist() if mx > 0 else v.tolist())
    return out


def _region_scores(preds: np.ndarray) -> dict:
    mean_act = np.abs(preds).mean(axis=0)
    scores = {}
    for region, (start, end) in REGION_MASKS.items():
        bounded_start = max(0, min(start, mean_act.shape[0]))
        bounded_end = max(bounded_start, min(end, mean_act.shape[0]))
        region_value = mean_act[bounded_start:bounded_end].mean() if bounded_end > bounded_start else 0.0
        scores[region] = round(float(np.clip(region_value * 500, 0, 100)), 1)
    scores['overall_impact'] = round(float(np.mean(list(scores.values()))), 1)
    return scores


def _activation_summary(preds: np.ndarray) -> dict:
    scores = _region_scores(preds)
    return {
        'scores': scores,
        'region_masks': REGION_MASKS,
        'peak_activation_step': int(np.abs(preds).mean(axis=1).argmax()),
    }


def _analysis_id(preds: np.ndarray, input_type: str, metadata: dict | None = None) -> str:
    hasher = hashlib.sha256()
    hasher.update(np.asarray(preds, dtype=np.float32).tobytes())
    hasher.update(input_type.encode('utf-8'))
    if metadata:
        hasher.update(repr(sorted(metadata.items())).encode('utf-8'))
    return hasher.hexdigest()[:20]


def _frame_for_viewer(preds: np.ndarray, step: int) -> np.ndarray:
    frames = np.asarray(preds, dtype=np.float64)
    safe_step = int(np.clip(step, 0, max(frames.shape[0] - 1, 0)))
    frame = np.abs(frames[safe_step]).reshape(-1)
    mx = float(frame.max()) if frame.size else 0.0
    if mx > 0:
        frame = frame / mx
    return frame


def _build_viewer_html(preds: np.ndarray, analysis_id: str, peak_step: int) -> str:
    if _NILEARN_IMPORT_ERROR:
        raise RuntimeError(f'nilearn plotting is unavailable: {_NILEARN_IMPORT_ERROR}')

    cached = _viewer_cache.get(analysis_id)
    if cached and os.path.exists(cached):
        return cached

    html_path = os.path.join(VIEWER_DIR, f'{analysis_id}.html')
    if os.path.exists(html_path):
        _viewer_cache[analysis_id] = html_path
        return html_path

    frame = _frame_for_viewer(preds, peak_step)
    if frame.size < 2:
        raise RuntimeError('Not enough vertices to render the cortical surface viewer.')

    half = frame.size // 2
    left_data = frame[:half]
    right_data = frame[half : half * 2]

    fsavg = datasets.fetch_surf_fsaverage(mesh='fsaverage5')
    left_view = plotting.view_surf(
        fsavg.infl_left,
        left_data,
        bg_map=fsavg.sulc_left,
        cmap='hot',
        black_bg=True,
        symmetric_cmap=False,
        colorbar=True,
        vmax=1.0,
    )
    right_view = plotting.view_surf(
        fsavg.infl_right,
        right_data,
        bg_map=fsavg.sulc_right,
        cmap='hot',
        black_bg=True,
        symmetric_cmap=False,
        colorbar=False,
        vmax=1.0,
    )

    left_embed = left_view._repr_html_()
    right_embed = right_view._repr_html_()
    combined_html = f"""<!doctype html>
<html>
  <head>
    <meta charset=\"utf-8\" />
    <title>TRIBE v2 Brain Viewer</title>
    <style>
      body {{ margin: 0; background: #05070c; color: #e8ecf1; font-family: Inter, Arial, sans-serif; }}
      .wrap {{ padding: 12px; }}
      .title {{ font-size: 16px; margin: 0 0 8px; }}
      .grid {{ display: grid; grid-template-columns: 1fr 1fr; gap: 10px; }}
      .card {{ background: #0b1020; border: 1px solid #1f2937; border-radius: 10px; overflow: hidden; }}
      .card iframe {{ width: 100%; min-height: 560px; border: 0; }}
      .meta {{ color: #9ca3af; font-size: 12px; margin-top: 8px; }}
    </style>
  </head>
  <body>
    <div class=\"wrap\">
      <p class=\"title\">TRIBE v2 Interactive Cortical Viewer</p>
      <div class=\"grid\">
        <div class=\"card\">{left_embed}</div>
        <div class=\"card\">{right_embed}</div>
      </div>
      <p class=\"meta\">analysis_id={analysis_id} · peak_activation_step={peak_step}</p>
    </div>
  </body>
</html>
"""

    with open(html_path, 'w', encoding='utf-8') as f:
        f.write(combined_html)

    _viewer_cache[analysis_id] = html_path
    return html_path


def _viewer_payload(preds: np.ndarray, input_type: str, metadata: dict, peak_step: int) -> dict:
    analysis_id = _analysis_id(preds, input_type, metadata)
    payload = {
        'analysis_id': analysis_id,
        'viewer_url': f'/viewer/{analysis_id}',
        'viewer_available': False,
    }
    try:
        _build_viewer_html(preds, analysis_id, peak_step)
        payload['viewer_available'] = True
    except Exception as e:
        payload['viewer_error'] = str(e)
    return payload


def _predict(events):
    preds, segments = tribe_model.predict(events=events)
    return np.asarray(preds), _segments_json(segments)


def _activation_payload(
    preds: np.ndarray,
    input_type: str,
    segments: list | None = None,
    metadata: dict | None = None,
) -> dict:
    activation = preds.tolist()
    summary = _activation_summary(preds)
    metadata = metadata or {}
    viewer = _viewer_payload(preds, input_type, metadata, summary['peak_activation_step'])

    payload = {
        'input_type': input_type,
        'shape': list(preds.shape),
        'activation': activation,
        'allPreds': _viewer_preds(preds),
        'segments': segments if segments is not None else [],
        'summary': summary,
        'metadata': metadata,
        # Convenience top-level fields for clients that don't want to unpack summary.
        'scores': summary['scores'],
        'region_masks': summary['region_masks'],
        'peak_activation_step': summary['peak_activation_step'],
        'analysis_id': viewer['analysis_id'],
        'viewer_url': viewer['viewer_url'],
        'viewer_available': viewer['viewer_available'],
    }
    if 'viewer_error' in viewer:
        payload['viewer_error'] = viewer['viewer_error']
    return payload


def _infer_text(text: str):
    normalized = text.strip()
    key = hashlib.sha256(normalized.encode('utf-8')).hexdigest()
    if key in _text_cache:
        return _text_cache[key]

    tmp = tempfile.NamedTemporaryFile(delete=False, suffix='.txt', mode='w', encoding='utf-8')
    try:
        tmp.write(normalized)
        tmp.flush()
        os.fsync(tmp.fileno())
        tmp.close()
        events = tribe_model.get_events_dataframe(text_path=tmp.name)
    finally:
        if os.path.exists(tmp.name):
            os.unlink(tmp.name)

    result = _predict(events)
    _text_cache[key] = result
    return result


async def _save_upload(file: UploadFile, allowed_ext: set[str]) -> str:
    ext = os.path.splitext(file.filename or '')[1].lower()
    if ext not in allowed_ext:
        raise HTTPException(400, f'Unsupported format {ext}. Allowed: {", ".join(sorted(allowed_ext))}')
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=ext)
    try:
        tmp.write(await file.read())
        tmp.flush()
        os.fsync(tmp.fileno())
        tmp.close()
        return tmp.name
    except Exception:
        if os.path.exists(tmp.name):
            os.unlink(tmp.name)
        raise


@app.get('/health')
async def health():
    return {
        'status': 'ok',
        'model': 'facebook/tribev2',
        'api_version': TRIBE_API_VERSION,
        'response_fields': [
            'activation',
            'allPreds',
            'segments',
            'shape',
            'input_type',
            'metadata',
            'summary',
            'scores',
            'region_masks',
            'peak_activation_step',
            'analysis_id',
            'viewer_url',
            'viewer_available',
        ],
        'text_cache_entries': len(_text_cache),
        'viewer_cache_entries': len(_viewer_cache),
        'viewer_backend': 'nilearn' if _NILEARN_IMPORT_ERROR is None else 'unavailable',
        'viewer_dir': VIEWER_DIR,
    }


class TextActivationRequest(BaseModel):
    text: str
    campaign_name: Optional[str] = None


@app.post('/activate/text')
async def activate_text(req: TextActivationRequest):
    if not req.text.strip():
        raise HTTPException(400, 'text cannot be empty')
    preds, segments = _infer_text(req.text)
    metadata = {'campaign_name': req.campaign_name or 'Unnamed Campaign'}
    return JSONResponse(_activation_payload(preds, 'text', segments, metadata))


@app.post('/activate/audio')
async def activate_audio(
    file: UploadFile = File(...),
    campaign_name: str = Form('Unnamed Campaign'),
):
    path = await _save_upload(file, AUDIO_EXT)
    try:
        events = tribe_model.get_events_dataframe(audio_path=path)
        preds, segments = _predict(events)
    finally:
        if os.path.exists(path):
            os.unlink(path)
    metadata = {'campaign_name': campaign_name, 'filename': file.filename}
    return JSONResponse(_activation_payload(preds, 'audio', segments, metadata))


@app.post('/activate/video')
async def activate_video(
    file: UploadFile = File(...),
    campaign_name: str = Form('Unnamed Campaign'),
):
    path = await _save_upload(file, VIDEO_EXT)
    try:
        events = tribe_model.get_events_dataframe(video_path=path)
        preds, segments = _predict(events)
    finally:
        if os.path.exists(path):
            os.unlink(path)
    metadata = {'campaign_name': campaign_name, 'filename': file.filename}
    return JSONResponse(_activation_payload(preds, 'video', segments, metadata))


@app.get('/viewer/{analysis_id}')
async def get_viewer(analysis_id: str):
    path = _viewer_cache.get(analysis_id)
    if not path:
        candidate = os.path.join(VIEWER_DIR, f'{analysis_id}.html')
        path = candidate if os.path.exists(candidate) else None
    if not path or not os.path.exists(path):
        raise HTTPException(404, f'Viewer not found for analysis_id={analysis_id}')
    return FileResponse(path, media_type='text/html')


print(f'API v{TRIBE_API_VERSION} — responses include activation, allPreds, summary, scores, peak_activation_step, analysis_id, viewer_url')
print('Endpoints: GET /health | GET /viewer/{analysis_id} | POST /activate/text | /activate/audio | /activate/video')

API v2.3 — responses include activation, allPreds, segments, shape, scores, peak_activation_step
Endpoints: GET /health | POST /activate/text | /activate/audio | /activate/video


In [4]:
import subprocess
import time

import nest_asyncio
import requests
import uvicorn
from pyngrok import ngrok
from threading import Thread

nest_asyncio.apply()

# Free port 8000 so re-running this cell picks up the latest API code
subprocess.run('fuser -k 8000/tcp 2>/dev/null || true', shell=True, check=False)
time.sleep(0.5)

for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

tunnel = ngrok.connect(8000, domain=NGROK_DOMAIN)
public_url = tunnel.public_url

Thread(
    target=lambda: uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning'),
    daemon=True,
).start()

time.sleep(1.5)
health = requests.get('http://127.0.0.1:8000/health', timeout=10).json()
expected_version = TRIBE_API_VERSION
if health.get('api_version') != expected_version:
    raise RuntimeError(
        f'Stale API on port 8000 (got {health.get("api_version")!r}, want {expected_version!r}). '
        'Restart kernel, then Run All from the install cell.'
    )

print('=' * 62)
print('TRIBE activation API is live')
print(f'  Base URL : {public_url}')
print(f'  API docs : {public_url}/docs')
print(f'  Version  : {health["api_version"]}  fields: {health["response_fields"]}')
print('=' * 62)
for ep in ['/health', '/activate/text', '/activate/audio', '/activate/video', '/viewer/{analysis_id}']:
    print(f'  {public_url}{ep}')
print('\nViewer test pattern: ' + public_url + '/viewer/<analysis_id from activate response>')
print('Set in frontend: VITE_TRIBE_API=' + public_url)
print('Server running — keep this cell alive.')

TRIBE activation API is live
  Base URL : https://sibling-luminous-gothic.ngrok-free.dev
  API docs : https://sibling-luminous-gothic.ngrok-free.dev/docs
  Version  : 2.3  fields: ['activation', 'allPreds', 'segments', 'shape', 'input_type', 'metadata', 'summary', 'scores', 'region_masks', 'peak_activation_step']
  https://sibling-luminous-gothic.ngrok-free.dev/health
  https://sibling-luminous-gothic.ngrok-free.dev/activate/text
  https://sibling-luminous-gothic.ngrok-free.dev/activate/audio
  https://sibling-luminous-gothic.ngrok-free.dev/activate/video

Set in frontend: VITE_TRIBE_API=https://sibling-luminous-gothic.ngrok-free.dev
Server running — keep this cell alive.


In [5]:
# Smoke test — saves FULL JSON for BrainViewer and validates interactive HTML viewer
import json
import os

import numpy as np
import requests
from IPython.display import FileLink, IFrame, display

base = 'http://127.0.0.1:8000'
health = requests.get(f'{base}/health', timeout=10).json()
print('Health:', health)
if health.get('api_version') != TRIBE_API_VERSION:
    raise RuntimeError(
        f'API version mismatch: {health.get("api_version")!r} != {TRIBE_API_VERSION!r}. '
        'Re-run the API cell + server cell (or Restart kernel → Run All).'
    )

r = requests.post(
    f'{base}/activate/text',
    json={'text': 'A short sentence for TRIBE.', 'campaign_name': 'Smoke Test Campaign'},
    timeout=1800,
)
r.raise_for_status()
result = r.json()


def _viewer_preds_from_activation(activation):
    """Per-frame 0–1 from |activation| (if API has no allPreds yet)."""
    arr = np.asarray(activation, dtype=np.float64)
    out = []
    for frame in arr:
        v = np.abs(frame)
        mx = float(v.max())
        out.append((v / mx).tolist() if mx > 0 else v.tolist())
    return out


required = (
    'activation',
    'allPreds',
    'segments',
    'shape',
    'input_type',
    'metadata',
    'summary',
    'scores',
    'peak_activation_step',
    'analysis_id',
    'viewer_url',
    'viewer_available',
)
missing = [k for k in required if k not in result]
if missing:
    if 'activation' in result and 'allPreds' not in result:
        result['allPreds'] = _viewer_preds_from_activation(result['activation'])
        result.setdefault('segments', [])
        result.setdefault('metadata', {})
        print('Warning: server missing allPreds — computed client-side; re-run API + server cells.')
    else:
        raise KeyError(f'API response missing fields: {missing}. Re-run API + server cells.')

api_path = '/kaggle/working/tribe_api_response.json'
with open(api_path, 'w', encoding='utf-8') as f:
    json.dump(result, f)

viewer_path = '/kaggle/working/brainviewer_payload.json'
viewer_payload = {
    'allPreds': result['allPreds'],
    'segments': result.get('segments', []),
    'shape': result['shape'],
    'input_type': result['input_type'],
    'metadata': result.get('metadata', {}),
    'campaign_name': result.get('metadata', {}).get('campaign_name'),
    'summary': result.get('summary', {}),
    'scores': result.get('scores', {}),
    'region_masks': result.get('region_masks', {}),
    'peak_activation_step': result.get('peak_activation_step'),
    'analysis_id': result.get('analysis_id'),
    'viewer_url': result.get('viewer_url'),
    'viewer_available': result.get('viewer_available', False),
}
with open(viewer_path, 'w', encoding='utf-8') as f:
    json.dump(viewer_payload, f)

# Compact binary
npy_path = '/kaggle/working/brainviewer_payload.npz'
np.savez_compressed(
    npy_path,
    allPreds=np.array(result['allPreds'], dtype=np.float32),
    activation=np.array(result['activation'], dtype=np.float32),
)

# Pull interactive HTML viewer from API and save as artifact
viewer_html_path = '/kaggle/working/brainviewer_interactive.html'
if result.get('viewer_available') and result.get('viewer_url'):
    viewer_resp = requests.get(f"{base}{result['viewer_url']}", timeout=300)
    viewer_resp.raise_for_status()
    with open(viewer_html_path, 'w', encoding='utf-8') as f:
        f.write(viewer_resp.text)
else:
    print('Warning: viewer generation unavailable:', result.get('viewer_error'))

print('input_type:', result['input_type'])
print('metadata:', result.get('metadata'))
print('shape:', result['shape'], '  (frames, vertices)')
print('segments:', len(result.get('segments', [])))
print('peak_activation_step:', result.get('peak_activation_step'))
print('analysis_id:', result.get('analysis_id'))
print('viewer_url:', result.get('viewer_url'))
print('viewer_available:', result.get('viewer_available'))
print('scores:', result.get('scores'))
print()

artifact_paths = [api_path, viewer_path, npy_path]
if os.path.exists(viewer_html_path):
    artifact_paths.append(viewer_html_path)

for path in artifact_paths:
    print(f'{path}  ({os.path.getsize(path) / 1e6:.2f} MB)')

print()
print('Download from Kaggle Output, or use in frontend:')
display(FileLink(api_path))
display(FileLink(viewer_path))
display(FileLink(npy_path))
if os.path.exists(viewer_html_path):
    display(FileLink(viewer_html_path))
    print('Inline preview (interactive WebGL):')
    display(IFrame(src=viewer_html_path, width='100%', height=720))

print('Smoke test passed — use viewer_url or brainviewer_interactive.html to validate visualization')

Health: {'status': 'ok', 'model': 'facebook/tribev2', 'api_version': '2.3', 'response_fields': ['activation', 'allPreds', 'segments', 'shape', 'input_type', 'metadata', 'summary', 'scores', 'region_masks', 'peak_activation_step'], 'text_cache_entries': 0}


INFO - Wrote TTS audio to /kaggle/working/tribe_cache/tribev2.demo_utils.TextToEvents.get_events,0/text=A-short-sentence-for-TRIBE.-019e54a3/audio.mp3
Extracting words from audio: 100%|██████████| 1/1 [02:00<00:00, 120.31s/it]
/usr/local/lib/python3.12/dist-packages/neuralset/events/utils.py:134: UserWarning: The events dataframe contains an `Index` column. This is dangerous, please add drop=True in calls to df.reset_index(). Dropping it automatically.
  warnings.warn(msg)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 65.6 MB/s  0:00:04
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


Add context to words: 100%|██████████| 5/5 [00:00<00:00, 28688.81it/s]
[00:23:51 WARNING] Removing extractor video as there are no corresponding events
[00:23:51 INFO] Preparing extractor: text
Computing word embeddings:   0%|          | 0/2 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

...

input_type: text
metadata: {'campaign_name': 'Smoke Test Campaign'}
shape: [3, 20484]   (frames, vertices)
segments: 3
peak_activation_step: 0
scores: {'visual_cortex': 40.5, 'language_network': 41.0, 'attention': 42.4, 'emotional_response': 39.3, 'memory_encoding': 38.0, 'overall_impact': 40.2}

/kaggle/working/tribe_api_response.json  (2.61 MB)
/kaggle/working/brainviewer_payload.json  (1.28 MB)
/kaggle/working/brainviewer_payload.npz  (0.45 MB)

Download from Kaggle Output, or use in frontend:


/kaggle/working/tribe_api_response.json

/kaggle/working/brainviewer_payload.json

/kaggle/working/brainviewer_payload.npz

Smoke test passed — use brainviewer_payload.json in BrainViewer


In [ ]:
# API + brainviewer_payload.json fields:
#   activation           — raw TRIBE output (frames × ~20484)
#   allPreds             — per-frame |value| normalized to 0–1 (pass to BrainViewer)
#   segments             — timeline segments from predict()
#   shape                — [n_frames, n_vertices]
#   metadata             — optional campaign_name / filename labels
#   scores               — approximate region scores from REGION_MASKS
#   peak_activation_step — frame with strongest mean absolute activation
#   summary              — scores + region_masks + peak_activation_step
#   analysis_id          — stable id for this activation result
#   viewer_url           — GET /viewer/{analysis_id} interactive HTML endpoint
#   viewer_available     — whether HTML viewer was generated successfully